# 🎨 Cartoonify — Canny (Secondary Mode)

Uses **Canny edge detection** as the ControlNet conditioning signal instead of a depth map.
Canny extracts the hard geometric outlines of the source photo — face contours, silhouettes,
clothing folds — and FLUX regenerates the image constrained to those edges.

**When to use this over Kontext (`09`):**
- The subject must be immediately recognisable as a specific person
- You want the cartoon to closely follow the source framing and proportions
- Portrait-focused caricature rather than full scene recomposition

**Pipeline:**
- **Gemini 2.5 Flash Lite** — story → structured prompt
- **OpenCV Canny** — extracts hard geometric edges
- **FLUX.1-dev + ControlNet-Union-Pro-2.0** — generates cartoon within edge structure
- **LoRA fine-tune** — cartoon/satirical style

**Requires:** Google Colab Pro → Runtime → Change runtime type → **A100 GPU**

---

## 1 — Confirm A100 GPU

In [ ]:
!nvidia-smi

## 2 — Install Dependencies

In [ ]:
!pip install --quiet diffusers transformers accelerate sentencepiece
!pip install --quiet gradio
!pip install --quiet huggingface_hub
!pip install --quiet google-genai
!pip install --quiet opencv-python-headless

In [ ]:
import os
os.kill(os.getpid(), 9)
print("IGNORE: session crashed. This is intentional — continue to the next cell.")

## 3 — Imports

In [ ]:
import gc
import datetime
import os
import numpy as np
import cv2
import torch
import gradio as gr
import google.genai as genai
import google.genai.types as genai_types
from PIL import Image
from diffusers import FluxControlNetPipeline, FluxControlNetModel

print(f'torch  : {torch.__version__}')
print(f'CUDA   : {torch.cuda.is_available()} — {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"}')
print(f'cv2    : {cv2.__version__}')
print(f'gradio : {gr.__version__}')
print(f'genai  : {genai.__version__}')

## 4 — Mount Google Drive

In [ ]:
import shutil
from google.colab import drive

if os.path.isdir('/content/drive') and os.listdir('/content/drive'):
    os.system('fusermount -u /content/drive 2>/dev/null || true')
    shutil.rmtree('/content/drive', ignore_errors=True)

drive.mount('/content/drive')

## 5 — API Tokens

**Hugging Face** — add a Read token to Colab Secrets as `HF_TOKEN`.

**Google Gemini** — add your key to Colab Secrets as `GOOGLE_API_KEY`
([aistudio.google.com/apikey](https://aistudio.google.com/apikey)).

In [ ]:
from google.colab import userdata
from huggingface_hub import login

HF_TOKEN       = userdata.get('HF_TOKEN')
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')

login(token=HF_TOKEN, add_to_git_credential=False)
print('✓ Logged in to Hugging Face')
print('✓ Google API key loaded' if GOOGLE_API_KEY else '⚠️  GOOGLE_API_KEY not found — story-to-prompt will not work')

## 6 — Configuration

| Variable | What it does |
|---|---|
| `LORA_SOURCE` | `"huggingface"` · `"drive"` |
| `LORA_DRIVE_PATH` | Path to your `.safetensors` on Drive |
| `DEFAULT_TRIGGER` | Keyword baked into training captions |
| `DEFAULT_CANNY_LOW` | Lower Canny threshold — raise to reduce edge density |
| `DEFAULT_CANNY_HIGH` | Upper Canny threshold — raise to suppress weak edges |
| `DEFAULT_CN_SCALE` | How strictly edges constrain the output (0.1–1.0) |
| `DEFAULT_CN_END` | Fraction of steps ControlNet stays active (0.0–1.0) |

In [ ]:
# ── LoRA source ────────────────────────────────────────────────────────────────
LORA_SOURCE      = 'drive'   # 'huggingface' | 'drive'

LORA_HF_REPO     = 'strangerzonehf/Ghibli-Flux-Cartoon-LoRA'
LORA_HF_WEIGHT   = 'Ghibili-Cartoon-Art.safetensors'

LORA_DRIVE_PATH  = '/content/drive/MyDrive/cartoonify/02_FLUX.1/weights_FLUX.1/gdo_cartoon/gdo_cartoon.safetensors'

# ── Trigger word ───────────────────────────────────────────────────────────────
DEFAULT_TRIGGER  = 'gdo_cartoon'

# ── Prompt default ─────────────────────────────────────────────────────────────
DEFAULT_PROMPT   = 'satirical cartoon illustration, bold outlines, vivid flat colours, expressive exaggerated features'

# ── Generation defaults ────────────────────────────────────────────────────────
DEFAULT_GUIDANCE  = 3.5
DEFAULT_STEPS     = 28
DEFAULT_CN_SCALE  = 0.7    # how strongly edges constrain the output
DEFAULT_CN_END    = 0.8    # ControlNet active for first 80% of steps, FLUX finishes freely
DEFAULT_SEED      = 42

# ── Canny thresholds ───────────────────────────────────────────────────────────
# Lower values = more edges = tighter structural constraint
# Higher values = fewer edges = looser, more expressive output
DEFAULT_CANNY_LOW  = 50    # edges weaker than this are discarded
DEFAULT_CANNY_HIGH = 200   # edges stronger than this are always kept

# ── Base models ────────────────────────────────────────────────────────────────
BASE_MODEL       = 'black-forest-labs/FLUX.1-dev'
CONTROLNET_MODEL = 'Shakker-Labs/FLUX.1-dev-ControlNet-Union-Pro-2.0'

# ── Gemini ─────────────────────────────────────────────────────────────────────
GOOGLE_MODEL = 'gemini-2.5-flash-lite'

# ── Output directory ───────────────────────────────────────────────────────────
OUTPUT_DIR = '/content/drive/MyDrive/cartoonify/outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f'LoRA source      : {LORA_SOURCE}')
print(f'LoRA             : {LORA_HF_REPO if LORA_SOURCE == "huggingface" else LORA_DRIVE_PATH}')
print(f'Trigger word     : "{DEFAULT_TRIGGER}"')
print(f'Canny thresholds : low={DEFAULT_CANNY_LOW}  high={DEFAULT_CANNY_HIGH}')
print(f'Gemini model     : {GOOGLE_MODEL}')
print(f'Output dir       : {OUTPUT_DIR}')

## 7 — Load Models

First run downloads FLUX.1-dev + ControlNet-Union-Pro-2.0 (~28 GB total).
Subsequent runs use the Colab cache — same weights as `07` and `08` if already cached.

⏱️ Expected time: **3–6 minutes** first run · **~1 minute** warm cache.

In [ ]:
for _name in ['pipe', 'controlnet']:
    if _name in globals():
        del globals()[_name]
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()
_free, _total = torch.cuda.mem_get_info()
print(f'GPU before loading: {_free/1e9:.1f} GB free / {_total/1e9:.1f} GB total\n')

print('Loading ControlNet...')
controlnet = FluxControlNetModel.from_pretrained(CONTROLNET_MODEL, torch_dtype=torch.bfloat16)
print('✓ ControlNet ready')

print('\nLoading FLUX.1-dev pipeline...')
pipe = FluxControlNetPipeline.from_pretrained(BASE_MODEL, controlnet=controlnet, torch_dtype=torch.bfloat16).to('cuda')
print('✓ FLUX pipeline ready')

import peft.import_utils as _peft_utils
import peft.tuners.lora.torchao as _peft_torchao
_orig = _peft_utils.is_torchao_available
def _safe():
    try: return _orig()
    except ImportError: return False
_peft_utils.is_torchao_available = _safe
_peft_torchao.is_torchao_available = _safe
print('✓ PEFT torchao patched')

print('\nLoading LoRA weights...')
if LORA_SOURCE == 'huggingface':
    pipe.load_lora_weights(LORA_HF_REPO, weight_name=LORA_HF_WEIGHT)
else:
    pipe.load_lora_weights(LORA_DRIVE_PATH)
print('✅ All models ready.')

## 8 — Story-to-Prompt (Gemini)

Unchanged from `08` and `09` — converts plain language into the seven-layer structured
prompt vocabulary the LoRA was trained on.

In [ ]:
GEMINI_SYSTEM_PROMPT = """You are a prompt engineer for a FLUX.1 LoRA fine-tuned on editorial and political cartoons.
Convert the user's story into a structured image generation prompt.

OUTPUT FORMAT — one line only, no explanation, no preamble:
gdo_cartoon <medium>, <technique>, <color>, <mood>, <commentary>, <composition>

RULES:
- Always start with: gdo_cartoon editorial cartoon | political cartoon | caricature | newspaper illustration
- Always include: pen and ink | cross-hatching | bold outlines | hand-drawn linework | varied line weight
- Default color: black and white | monochrome | print cartoon | white background
  (use spot color red only if violence or urgency is central to the story)
- Derive mood, commentary, and composition from the story
- Pipe-separated keywords within each layer; comma between layers
- Max 5 keywords per layer
- Composition: estimate figure count, suggest layout, add speech bubble hint if there is dialogue

EXAMPLES:

Story: A politician hands out empty promises while people queue for food.
Prompt: gdo_cartoon editorial cartoon | political cartoon | caricature | newspaper illustration, pen and ink | cross-hatching | bold outlines | hand-drawn linework | varied line weight, black and white | monochrome | print cartoon | white background, satirical | ironic | dark humour | critical | bleak, political commentary | scathing | power critique | deadpan | accusatory, two-group layout | standing figure left | queue of figures right | speech bubble top | eye level | wide shot

Story: A general sits behind a massive desk covered in medals while tiny soldiers march across a map below him.
Prompt: gdo_cartoon editorial cartoon | political cartoon | caricature | newspaper illustration, pen and ink | cross-hatching | bold outlines | hand-drawn linework | varied line weight, black and white | monochrome | print cartoon | white background, absurdist | dark | pompous | grotesque | confrontational, political condemnation | war metaphor | accusatory | ironic | scathing, large figure dominates upper frame | tiny figures lower third | desk as stage | top-heavy composition | eye level

Story: Two leaders shake hands in front of cameras while their shadows fight each other behind them.
Prompt: gdo_cartoon editorial cartoon | political cartoon | caricature | newspaper illustration, pen and ink | cross-hatching | bold outlines | hand-drawn linework | varied line weight, black and white | monochrome | print cartoon | white background, satirical | ironic | duplicitous | tense | darkly comic, political commentary | diplomatic hypocrisy | sharp | deadpan | allegorical, two figure layout | shadow duality | frontal handshake foreground | fighting silhouettes background | centred composition | eye level"""


def build_prompt_from_story(story: str, trigger_word: str) -> str:
    if not story.strip():
        raise gr.Error('Please enter a story before building a prompt.')
    if not GOOGLE_API_KEY:
        raise gr.Error('GOOGLE_API_KEY not found in Colab Secrets.')

    client = genai.Client(api_key=GOOGLE_API_KEY)

    response = client.models.generate_content(
        model=GOOGLE_MODEL,
        contents=f'Story: {story.strip()}',
        config=genai_types.GenerateContentConfig(
            system_instruction=GEMINI_SYSTEM_PROMPT,
            temperature=0.7,
            max_output_tokens=300,
        ),
    )

    prompt = response.text.strip()

    active_trigger = trigger_word.strip()
    if active_trigger and active_trigger != 'gdo_cartoon':
        prompt = prompt.replace('gdo_cartoon', active_trigger, 1)

    return prompt


print('✓ Gemini story-to-prompt function ready')

## 9 — Launch Interface

**Canny thresholds explained:**
- **Low threshold** — edges weaker than this are discarded entirely
- **High threshold** — edges stronger than this are always kept
- Edges between the two values are kept only if connected to a strong edge
- Lower both values → more edges → output stays closer to the source geometry
- Raise both values → fewer edges → more stylistic freedom within the structure

Re-run this cell to restart the UI without reloading models.

In [ ]:
# ── Canny extraction helper ────────────────────────────────────────────────────

def extract_canny(pil_image: Image.Image, low: int, high: int) -> Image.Image:
    img_np = np.array(pil_image)
    edges  = cv2.Canny(img_np, low, high)
    edges  = edges[:, :, None]
    edges  = np.concatenate([edges, edges, edges], axis=2)
    return Image.fromarray(edges)


# ── Inference function ─────────────────────────────────────────────────────────

def cartoonify(
    image,
    prompt,
    trigger_word,
    guidance_scale,
    num_steps,
    cn_scale,
    cn_end,
    canny_low,
    canny_high,
    seed,
    progress=gr.Progress(track_tqdm=True),
):
    if image is None:
        raise gr.Error('Please upload an image first.')

    trigger = trigger_word.strip()
    style   = prompt.strip()
    full_prompt = f'{trigger} {style}'.strip() if trigger else style

    src = Image.fromarray(image).convert('RGB')
    src = src.resize((1024, 1024), Image.LANCZOS)

    progress(0.0, desc='Extracting Canny edges...')
    canny_image = extract_canny(src, int(canny_low), int(canny_high))

    progress(0.1, desc='Running FLUX Canny inference (~30 seconds)...')
    generator = torch.Generator(device='cuda').manual_seed(int(seed))

    with torch.inference_mode():
        result = pipe(
            prompt=full_prompt,
            control_image=canny_image,
            control_mode=0,                               # 0 = Canny in Union Pro
            width=1024,
            height=1024,
            num_inference_steps=int(num_steps),
            guidance_scale=float(guidance_scale),
            controlnet_conditioning_scale=float(cn_scale),
            control_guidance_end=float(cn_end),
            generator=generator,
            joint_attention_kwargs={'scale': 1.0},
            max_sequence_length=512,
        ).images[0]

    ts = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
    save_path = f'{OUTPUT_DIR}/{ts}_cartoonify_canny.png'
    result.save(save_path)
    print(f'Saved: {save_path}')

    gc.collect()
    torch.cuda.empty_cache()

    return canny_image, result

In [ ]:
CSS = '''
.gradio-container {
    font-family: "Inter", -apple-system, BlinkMacSystemFont, sans-serif !important;
    background: #f0eff5 !important;
    max-width: 1280px !important;
    margin: 0 auto !important;
    padding: 1.5rem !important;
}
#app-header {
    background: linear-gradient(135deg, #0f0c29 0%, #302b63 55%, #24243e 100%);
    border-radius: 18px;
    padding: 2.5rem 2rem 2.2rem;
    text-align: center;
    margin-bottom: 1.5rem;
    box-shadow: 0 8px 30px rgba(48, 43, 99, 0.35);
}
.app-title {
    font-size: 3rem; font-weight: 900; letter-spacing: -2px;
    color: #ffffff; margin: 0 0 0.5rem 0; line-height: 1; display: block;
}
.app-subtitle {
    font-size: 1rem; color: #c4b5fd; margin: 0;
    font-weight: 400; line-height: 1.6; display: block;
}
.app-badge {
    display: inline-block; margin-top: 1rem;
    background: rgba(255,255,255,0.12); border: 1px solid rgba(255,255,255,0.2);
    border-radius: 999px; padding: 0.25rem 0.85rem;
    font-size: 0.75rem; color: #e9d5ff;
    letter-spacing: 0.05em; font-weight: 600; text-transform: uppercase;
}
.input-card, .output-card {
    background: #ffffff; border-radius: 16px;
    padding: 1.5rem 1.5rem 1.75rem;
    box-shadow: 0 1px 4px rgba(0,0,0,0.07); border: 1px solid #e5e7eb;
}
.section-label {
    display: block; font-size: 0.68rem; font-weight: 800;
    text-transform: uppercase; letter-spacing: 0.12em;
    color: #7c3aed; margin-bottom: 0.6rem; margin-top: 1.1rem;
}
.section-label:first-child { margin-top: 0; }
.upload-zone {
    border: 2.5px dashed #7c3aed !important; border-radius: 14px !important;
    background: #faf5ff !important;
    transition: border-color 0.2s, background 0.2s !important;
}
.upload-zone:hover { border-color: #5b21b6 !important; background: #f5edff !important; }
.story-box {
    border: 2px solid #ddd6fe !important; border-radius: 12px !important;
    background: #faf5ff !important;
}
.story-box:focus-within { border-color: #7c3aed !important; }
#build-btn {
    background: linear-gradient(135deg, #059669 0%, #047857 100%) !important;
    color: white !important; border: none !important; border-radius: 10px !important;
    font-size: 0.95rem !important; font-weight: 700 !important;
    height: 44px !important; width: 100% !important; cursor: pointer !important;
    box-shadow: 0 3px 12px rgba(5,150,105,0.35) !important;
    transition: opacity 0.2s, transform 0.15s !important; margin-top: 0.6rem !important;
}
#build-btn:hover  { opacity: 0.90 !important; transform: translateY(-1px) !important; }
#build-btn:active { transform: translateY(0) !important; }
#build-btn:disabled { opacity: 0.5 !important; cursor: wait !important; }
#generate-btn {
    background: linear-gradient(135deg, #7c3aed 0%, #5b21b6 100%) !important;
    color: white !important; border: none !important; border-radius: 12px !important;
    font-size: 1.05rem !important; font-weight: 700 !important;
    height: 54px !important; width: 100% !important; letter-spacing: 0.02em !important;
    cursor: pointer !important; box-shadow: 0 4px 18px rgba(124,58,237,0.38) !important;
    transition: opacity 0.2s, transform 0.15s !important; margin-top: 1rem !important;
}
#generate-btn:hover  { opacity: 0.90 !important; transform: translateY(-1px) !important; }
#generate-btn:active { transform: translateY(0) !important; }
#generate-btn:disabled { opacity: 0.5 !important; cursor: wait !important; }
.output-image {
    border-radius: 12px !important; overflow: hidden !important; background: #f9fafb !important;
}
.tip-box {
    background: #f5f3ff; border: 1px solid #ddd6fe; border-radius: 10px;
    padding: 0.75rem 1rem; font-size: 0.82rem; color: #5b21b6;
    line-height: 1.5; margin-bottom: 0.75rem;
}
.tip-box-green {
    background: #f0fdf4; border: 1px solid #bbf7d0; border-radius: 10px;
    padding: 0.75rem 1rem; font-size: 0.82rem; color: #065f46;
    line-height: 1.5; margin-bottom: 0.75rem;
}
.or-divider {
    display: flex; align-items: center; gap: 0.75rem;
    margin: 0.9rem 0 0.5rem; color: #9ca3af;
    font-size: 0.75rem; font-weight: 600;
    letter-spacing: 0.08em; text-transform: uppercase;
}
.or-divider::before, .or-divider::after {
    content: ""; flex: 1; height: 1px; background: #e5e7eb;
}
footer { display: none !important; }
'''


with gr.Blocks(css=CSS, theme=gr.themes.Soft(), title='Cartoonify — Canny') as demo:

    gr.HTML('''
        <div id="app-header">
            <span class="app-title">🎨 Cartoonify</span>
            <span class="app-subtitle">
                Tell a story &mdash; get a satirical cartoon.<br>
                Gemini &plus; FLUX.1-dev &plus; Canny ControlNet &plus; LoRA
            </span>
            <span class="app-badge">Secondary Mode &nbsp;&middot;&nbsp; Canny</span>
        </div>
    ''')

    with gr.Row(equal_height=False, variant='panel'):

        # ── Left column — Inputs ───────────────────────────────────────────────
        with gr.Column(scale=5, min_width=360, elem_classes=['input-card']):

            gr.HTML('<span class="section-label">📤 Source Image</span>')
            img_input = gr.Image(
                label='Drag & drop a photo here, or click to browse',
                type='numpy',
                elem_classes=['upload-zone'],
                sources=['upload', 'clipboard'],
                height=250,
            )

            with gr.Accordion('📖 What\'s the Story?  (click to expand)', open=False):
                gr.HTML('<div class="tip-box-green">'
                        '✨ Describe what you want to say in plain language. '
                        'Gemini converts it into a structured prompt that matches '
                        'the LoRA training vocabulary — then you can edit it before generating.'
                        '</div>')
                story_input = gr.Textbox(
                    label='Your story',
                    placeholder=(
                        'e.g. A politician hands out empty promises while people queue for food.\n'
                        'A general sits behind a massive desk while tiny soldiers march across a map below him.'
                    ),
                    lines=4,
                    max_lines=8,
                    elem_classes=['story-box'],
                )
                build_btn = gr.Button(
                    '✨  Build Prompt from Story',
                    variant='secondary',
                    elem_id='build-btn',
                )

            gr.HTML('<div class="or-divider">or write a prompt directly</div>')

            gr.HTML('<span class="section-label">✍️ Style Prompt</span>')
            prompt_input = gr.Textbox(
                label='Structured prompt (pre-filled by Gemini, or write your own)',
                placeholder='e.g. satirical cartoon, bold outlines, vivid flat colours, exaggerated expressions',
                value=DEFAULT_PROMPT,
                lines=3,
                max_lines=6,
            )

            gr.HTML('<span class="section-label">🔑 LoRA Trigger Word</span>')
            trigger_input = gr.Textbox(
                label='Trigger word (editable — no restart needed)',
                placeholder='e.g. gdo_cartoon — leave blank if your LoRA has no trigger',
                value=DEFAULT_TRIGGER,
                lines=1,
                info='The keyword baked into your LoRA training captions. Gemini uses this automatically.',
            )

            gr.HTML('<span class="section-label">⚙️ Advanced Controls</span>')
            with gr.Accordion('Expand to adjust generation parameters', open=False):
                gr.HTML('<div class="tip-box">'
                        '💡 <strong>Canny thresholds</strong> control how many edges are extracted. '
                        'Lower values → denser edges → output stays closer to the source geometry. '
                        'Raise both values to give FLUX more freedom within the structure.'
                        '</div>')
                canny_low_slider = gr.Slider(
                    minimum=0, maximum=200, value=DEFAULT_CANNY_LOW,
                    step=10, label='Canny Low Threshold',
                    info='Edges weaker than this are discarded. Lower = more edges.'
                )
                canny_high_slider = gr.Slider(
                    minimum=50, maximum=500, value=DEFAULT_CANNY_HIGH,
                    step=10, label='Canny High Threshold',
                    info='Edges stronger than this are always kept. Higher = fewer, stronger edges only.'
                )
                guidance_slider = gr.Slider(
                    minimum=1.0, maximum=15.0, value=DEFAULT_GUIDANCE,
                    step=0.5, label='Guidance Scale',
                    info='How strongly the prompt steers the output. 3–5 is the FLUX sweet spot.',
                )
                steps_slider = gr.Slider(
                    minimum=10, maximum=50, value=DEFAULT_STEPS,
                    step=1, label='Inference Steps',
                )
                cn_scale_slider = gr.Slider(
                    minimum=0.1, maximum=1.5, value=DEFAULT_CN_SCALE,
                    step=0.05, label='ControlNet Scale',
                    info='How strongly edges constrain the output. Lower = more creative freedom.',
                )
                cn_end_slider = gr.Slider(
                    minimum=0.3, maximum=1.0, value=DEFAULT_CN_END,
                    step=0.05, label='ControlNet Guidance End',
                    info='Fraction of steps where ControlNet stays active. 0.8 = active for first 80%, FLUX finishes freely.',
                )
                seed_input = gr.Number(
                    value=DEFAULT_SEED, label='Seed',
                    precision=0,
                    info='Same seed + same inputs = identical result.',
                )

            generate_btn = gr.Button(
                '🎨  Cartoonify  →',
                variant='primary',
                elem_id='generate-btn',
            )

        # ── Right column — Outputs ─────────────────────────────────────────────
        with gr.Column(scale=6, min_width=480, elem_classes=['output-card']):

            gr.HTML('<span class="section-label">🎭 Cartoonified Result</span>')
            result_output = gr.Image(
                label='Output  ·  Use the ↓ button to download',
                type='pil',
                elem_classes=['output-image'],
                show_download_button=True,
                height=480,
                interactive=False,
            )

            with gr.Accordion('🖊️ Canny Edge Map  (technical preview)', open=False):
                gr.HTML('<div class="tip-box" style="margin-top:0.5rem;">'
                        'Hard geometric edges extracted from the source photo. '
                        'FLUX generates the cartoon constrained to stay within these lines. '
                        'Adjust the Canny thresholds above to control edge density.'
                        '</div>')
                canny_output = gr.Image(
                    label='Canny edge map',
                    type='pil',
                    elem_classes=['output-image'],
                    interactive=False,
                    height=220,
                )

    # ── Button wiring ──────────────────────────────────────────────────────────
    build_btn.click(
        fn=build_prompt_from_story,
        inputs=[story_input, trigger_input],
        outputs=[prompt_input],
        api_name=False,
    )

    generate_btn.click(
        fn=cartoonify,
        inputs=[
            img_input,
            prompt_input,
            trigger_input,
            guidance_slider,
            steps_slider,
            cn_scale_slider,
            cn_end_slider,
            canny_low_slider,
            canny_high_slider,
            seed_input,
        ],
        outputs=[canny_output, result_output],
        api_name='cartoonify',
    )


demo.launch(share=True, show_error=True, quiet=False)